In [16]:
# ==============================================================================
# 解析結果の診断：ファイル名救済ロジック追加版
# ==============================================================================

library(dplyr)
library(stringr)
library(tidyr)

# --- 設定 ---
root_dir <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/"
target_folder_date <- "20251217"

# モデル名リスト（表記揺れも含めて検索対象にする）
known_model_names <- c(
  "SVMRadial", "svmLinear", "SVM", 
  "gaussPrRadial", "gaussprRadial", "GPR",
  "ppr", "PPR",
  "gcvEarth", "MARS",
  "knn", "KNN",
  "PLS", 
  "Rborist", "RF",
  "monmlp", "MonMLP" # MonMLPを追加
)

thresholds <- list("Jsc"=2.0, "Voc"=0.1, "FF"=0.1, "PCEmax"=1.0)

known_datasets <- c(
  "n_base_OH_rebuilt", "n_base_FP_rebuilt", "n_base",
  "n_all_OH_rebuilt",  "n_all_FP_rebuilt",  "n_all",
  "m_base_OH_rebuilt", "m_base_FP_rebuilt", "m_base",
  "m_all_OH_rebuilt",  "m_all_FP_rebuilt",  "m_all"
)
known_datasets <- known_datasets[order(nchar(known_datasets), decreasing = TRUE)]
known_targets <- c("Jsc", "Voc", "FF", "PCEmax")

# --- 実行 ---

if (!dir.exists(root_dir)) stop("ディレクトリが見つかりません")
all_files <- list.files(path = root_dir, pattern = "_pred.csv$", recursive = TRUE, full.names = TRUE)
target_files <- all_files[str_detect(all_files, target_folder_date)]

cat(sprintf("Found %d files. Processing...\n", length(target_files)))

bad_list <- data.frame()

for (f_path in target_files) {
  fname <- basename(f_path)
  
  # === 1. モデル名の特定 (強化版) ===
  model_name <- "Unknown_Model"
  
  # A. まずファイル名の中に、既知のモデル名が含まれているかチェック (最優先)
  #    例: fixed_..._MonMLP_... -> "MonMLP"
  for (known in known_model_names) {
    # ファイル名の中にモデル名が含まれているか (大文字小文字無視はあえてしない)
    if (str_detect(fname, fixed(known, ignore_case = TRUE))) {
      model_name <- known
      break
    }
  }
  
  # B. ファイル名になければ、フォルダ構造から探す (バックアップ)
  if (model_name == "Unknown_Model") {
    path_parts <- str_split(f_path, "/")[[1]]
    for (known in known_model_names) {
      if (any(str_detect(path_parts, fixed(known, ignore_case = TRUE)))) {
        model_name <- known
        break
      }
    }
  }

  # === 2. ターゲット・データセット特定 ===
  target_var <- NA
  for (tgt in known_targets) {
    if (str_detect(fname, paste0("_", tgt, "_pred.csv"))) {
      target_var <- tgt
      break
    }
  }
  if (is.na(target_var)) next 
  
  dataset_name <- NA
  for (ds in known_datasets) {
    if (str_detect(fname, ds)) {
      dataset_name <- ds
      break
    }
  }
  if (is.na(dataset_name)) dataset_name <- "Unknown_Data"
  
  # === 3. データ処理 ===
  thresh_val <- thresholds[[target_var]]
  df <- read.csv(f_path, check.names = FALSE)
  if (!all(c("Observed", "Predicted", "SampleID") %in% colnames(df))) next
  
  df$Residual <- df$Observed - df$Predicted
  df$AbsError <- abs(df$Residual)
  df$APE <- ifelse(df$Observed == 0, NA, (df$AbsError / abs(df$Observed)) * 100)
  
  df$Direction <- ifelse(df$Residual > 0, 
                         "Underestimated (Pred < Obs)", 
                         "Overestimated (Pred > Obs)")
  
  bad_samples <- df %>% filter(AbsError > thresh_val)
  
  if (nrow(bad_samples) > 0) {
    temp_res <- data.frame(
      Model = model_name,
      Dataset = dataset_name,
      Target = target_var,
      SampleID = bad_samples$SampleID,
      Type = bad_samples$Type,
      Observed = bad_samples$Observed,
      Predicted = bad_samples$Predicted,
      Residual = bad_samples$Residual,
      AbsError = bad_samples$AbsError,
      APE_Percent = round(bad_samples$APE, 2),
      Direction = bad_samples$Direction,
      SourceFile = fname
    )
    bad_list <- rbind(bad_list, temp_res)
  }
}

# --- 保存 ---
if (nrow(bad_list) > 0) {
  
  # ファイル名特定救済ロジックの結果を反映して保存
  output_csv_detail <- file.path(root_dir, paste0(target_folder_date, "_Integrated_Bad_Predictions_Fixed.csv"))
  output_csv_summary <- file.path(root_dir, paste0(target_folder_date, "_Summary_Worst_SampleIDs_Fixed.csv"))
  
  # 並べ替えと列選択
  bad_list <- bad_list %>% 
    select(SampleID, Target, AbsError, APE_Percent, Direction, Model, Dataset, Observed, Predicted, Residual, Type, SourceFile) %>%
    arrange(desc(AbsError))
  
  write.csv(bad_list, output_csv_detail, row.names = FALSE)
  
  # 集計
  summary_table <- bad_list %>%
    group_by(SampleID) %>%
    summarise(
      Total_Failures = n(),
      Mean_APE = mean(APE_Percent, na.rm=TRUE),
      Models = paste(unique(Model), collapse=", ")
    ) %>%
    arrange(desc(Total_Failures))
  
  write.csv(summary_table, output_csv_summary, row.names = FALSE)
  
  cat("=== Analysis Complete (Model Name Fixed) ===\n")
  cat(sprintf("Results saved to: %s\n", output_csv_detail))
  
} else {
  cat("No bad predictions found.\n")
}

Found 480 files. Processing...
=== Analysis Complete (Model Name Fixed) ===
Results saved to: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression//20251217_Integrated_Bad_Predictions_Fixed.csv


In [17]:
# ==============================================================================
# 予測失敗サンプルの傾向分析ツール (全内訳カウント出力版)
# ==============================================================================

library(dplyr)
library(tidyr)
library(stringr)

# --- 設定 ---
root_dir <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/"
target_date <- "20251217"

# --- ファイル読み込み ---
pattern_str <- paste0(target_date, "_Integrated_Bad_Predictions_Fixed.csv")
target_file <- list.files(path = root_dir, pattern = pattern_str, recursive = TRUE, full.names = TRUE)

if (length(target_file) == 0) {
  pattern_str_bk <- paste0(target_date, "_Integrated_Bad_Predictions.csv")
  target_file <- list.files(path = root_dir, pattern = pattern_str_bk, recursive = TRUE, full.names = TRUE)
}
if (length(target_file) == 0) stop("詳細データファイルが見つかりません。")

target_file <- target_file[1]
cat(sprintf("Reading: %s\n", basename(target_file)))
df <- read.csv(target_file, check.names = FALSE)

# --- 全サンプルの抽出 ---
worst_ranking <- df %>% 
  count(SampleID) %>% 
  arrange(desc(n))

target_samples <- worst_ranking$SampleID
cat(sprintf("Analyzing detailed breakdown for %d samples...\n", length(target_samples)))

df_worst <- df

# ==============================================================================
# 視点1: Target (変数の傾向 + カウント)
# ==============================================================================
# カウント集計 (列名に Tgt_ をつける)
cross_target <- df_worst %>%
  count(SampleID, Target) %>%
  pivot_wider(names_from = Target, values_from = n, values_fill = 0, names_prefix = "Tgt_")

# 傾向判定 (Tgt_列を使って判定)
target_cols <- grep("^Tgt_", names(cross_target), value = TRUE)
cross_target$Trend_Target <- apply(cross_target[, target_cols, drop=FALSE], 1, function(x) {
  # 接頭辞を除去して元の名前に戻す
  raw_names <- gsub("^Tgt_", "", names(x))
  cols <- raw_names[x > 0]
  
  if (length(cols) == 4) return("All Targets")
  if ("PCEmax" %in% cols && length(cols) == 1) return("PCE Only")
  return(paste(cols, collapse = "+"))
})

# ==============================================================================
# 視点2: Model (モデルの傾向 + カウント)
# ==============================================================================
# カウント集計 (列名に Mod_ をつける)
cross_model <- df_worst %>%
  count(SampleID, Model) %>%
  pivot_wider(names_from = Model, values_from = n, values_fill = 0, names_prefix = "Mod_")

# 傾向判定
linear_models <- c("PLS", "svmLinear")
nonlinear_models <- c("MonMLP", "SVMRadial", "gaussPrRadial", "Rborist", "gcvEarth", "ppr", "monmlp", "knn")

model_cols <- grep("^Mod_", names(cross_model), value = TRUE)
cross_model$Trend_Model <- apply(cross_model[, model_cols, drop=FALSE], 1, function(x) {
  raw_names <- gsub("^Mod_", "", names(x))
  failed <- raw_names[x > 0]
  
  n_lin <- sum(failed %in% linear_models)
  n_non <- sum(failed %in% nonlinear_models)
  
  if (length(failed) >= 8) return("All Models")
  if (n_lin > 0 && n_non == 0) return("Linear Only")
  return("Mixed")
})

# ==============================================================================
# 視点3: Dataset (データセットの傾向 + カウント)
# ==============================================================================
# カウント集計 (列名に Dat_ をつける)
cross_dataset <- df_worst %>%
  count(SampleID, Dataset) %>%
  pivot_wider(names_from = Dataset, values_from = n, values_fill = 0, names_prefix = "Dat_")

# 傾向判定
dataset_cols <- grep("^Dat_", names(cross_dataset), value = TRUE)
cross_dataset$Trend_Dataset <- apply(cross_dataset[, dataset_cols, drop=FALSE], 1, function(x) {
  raw_names <- gsub("^Dat_", "", names(x))
  failed_ds <- raw_names[x > 0]
  n_failed <- length(failed_ds)
  
  if (n_failed >= 10) return("All Datasets")
  
  has_base <- any(str_detect(failed_ds, "base"))
  has_all  <- any(str_detect(failed_ds, "all"))
  if (has_base && !has_all) return("Base Only (Allで改善)")
  
  has_oh <- any(str_detect(failed_ds, "OH"))
  has_fp <- any(str_detect(failed_ds, "FP"))
  if (has_fp && !has_oh) return("FP Only (OHで改善)")
  if (has_oh && !has_fp) return("OH Only (FPで改善)")
  
  return("Mixed / Specific")
})

# ==============================================================================
# 視点4: Direction (方向性)
# ==============================================================================
df_worst$Dir_Simp <- ifelse(str_detect(df_worst$Direction, "Over"), "Over", "Under")
cross_dir <- df_worst %>%
  count(SampleID, Dir_Simp) %>%
  pivot_wider(names_from = Dir_Simp, values_from = n, values_fill = 0)

cross_dir$Trend_Dir <- apply(cross_dir[, -1], 1, function(x) {
  over <- ifelse(is.na(x["Over"]), 0, x["Over"])
  under <- ifelse(is.na(x["Under"]), 0, x["Under"])
  if (over == (over + under)) return("Always Over")
  if (under == (over + under)) return("Always Under")
  if (over > under * 2) return("Mostly Over")
  if (under > over * 2) return("Mostly Under")
  return("Unstable")
})

# ==============================================================================
# 統合レポート作成 (全カウント列を結合)
# ==============================================================================
final_report <- worst_ranking %>%
  left_join(cross_target, by = "SampleID") %>%   # Trend + 各Targetカウント
  left_join(cross_model, by = "SampleID") %>%    # Trend + 各Modelカウント
  left_join(cross_dataset, by = "SampleID") %>%  # Trend + 各Datasetカウント
  left_join(cross_dir %>% select(SampleID, Trend_Dir), by = "SampleID")

# 列の並び順を整理 (ID, 総数, 傾向まとめ, 詳細カウント...)
final_report <- final_report %>%
  select(
    SampleID, n, 
    Trend_Target, Trend_Model, Trend_Dataset, Trend_Dir, # まず傾向の要約
    starts_with("Tgt_"),  # Targetの詳細カウント
    starts_with("Mod_"),  # Modelの詳細カウント
    starts_with("Dat_")   # Datasetの詳細カウント
  )

# 保存
output_file <- file.path(root_dir, paste0(target_date, "_Analysis_Trend_Report_Detailed_Count.csv"))
write.csv(final_report, output_file, row.names = FALSE)

cat("\n=== Analysis Complete ===\n")
cat(sprintf("Analyzed samples: %d\n", nrow(final_report)))
cat(sprintf("Report saved to: %s\n", output_file))

# コンソール確認用 (最初の1行だけ構造を表示)
print(final_report[1, 1:10])

Reading: 20251217_Integrated_Bad_Predictions_Fixed.csv
Analyzing detailed breakdown for 1035 samples...

=== Analysis Complete ===
Analyzed samples: 1035
Report saved to: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression//20251217_Analysis_Trend_Report_Detailed_Count.csv
  SampleID   n Trend_Target Trend_Model    Trend_Dataset   Trend_Dir Tgt_FF
1      354 227  All Targets  All Models Mixed / Specific Mostly Over     87
  Tgt_Jsc Tgt_PCEmax Tgt_Voc
1      63         64      13


In [18]:
# ==============================================================================
# 解析ツール完全版：内装(Inner)と外挿(Outer)の自動分割＆傾向分析
# ==============================================================================

library(dplyr)
library(stringr)
library(tidyr)

# --- 設定 ---
root_dir <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/"
target_folder_date <- "20251217"

# 出力先フォルダを新しく作成（混ざらないように分離）
output_folder_name <- paste0(target_folder_date, "_Analysis_Split_Inner_Outer")
output_dir <- file.path(root_dir, output_folder_name)
if (!dir.exists(output_dir)) {
  dir.create(output_dir)
  cat(sprintf("Created new directory: %s\n", output_dir))
}

# 閾値設定
thresholds <- list("Jsc"=2.0, "Voc"=0.1, "FF"=0.1, "PCEmax"=1.0)

# モデル名・データセット定義
known_model_names <- c(
  "SVMRadial", "svmLinear", "SVM", 
  "gaussPrRadial", "gaussprRadial", "GPR",
  "ppr", "PPR",
  "gcvEarth", "MARS",
  "knn", "KNN",
  "PLS", 
  "Rborist", "RF",
  "monmlp", "MonMLP"
)

known_datasets <- c(
  "n_base_OH_rebuilt", "n_base_FP_rebuilt", "n_base",
  "n_all_OH_rebuilt",  "n_all_FP_rebuilt",  "n_all",
  "m_base_OH_rebuilt", "m_base_FP_rebuilt", "m_base",
  "m_all_OH_rebuilt",  "m_all_FP_rebuilt",  "m_all"
)
known_datasets <- known_datasets[order(nchar(known_datasets), decreasing = TRUE)]
known_targets <- c("Jsc", "Voc", "FF", "PCEmax")

# ==============================================================================
# PHASE 1: ファイル読み込みとバッドデータの抽出・統合
# ==============================================================================

cat("=== Phase 1: Scanning and Aggregating Data ===\n")

if (!dir.exists(root_dir)) stop("ルートディレクトリが見つかりません")
all_files <- list.files(path = root_dir, pattern = "_pred.csv$", recursive = TRUE, full.names = TRUE)
target_files <- all_files[str_detect(all_files, target_folder_date)]

cat(sprintf("Found %d files. Processing...\n", length(target_files)))

bad_list_all <- data.frame()

for (f_path in target_files) {
  fname <- basename(f_path)
  
  # 1. モデル名特定
  model_name <- "Unknown_Model"
  for (known in known_model_names) {
    if (str_detect(fname, fixed(known, ignore_case = TRUE))) {
      model_name <- known
      break
    }
  }
  if (model_name == "Unknown_Model") {
    path_parts <- str_split(f_path, "/")[[1]]
    for (known in known_model_names) {
      if (any(str_detect(path_parts, fixed(known, ignore_case = TRUE)))) {
        model_name <- known
        break
      }
    }
  }

  # 2. ターゲット・データセット特定
  target_var <- NA
  for (tgt in known_targets) {
    if (str_detect(fname, paste0("_", tgt, "_pred.csv"))) {
      target_var <- tgt
      break
    }
  }
  if (is.na(target_var)) next 
  
  dataset_name <- NA
  for (ds in known_datasets) {
    if (str_detect(fname, ds)) {
      dataset_name <- ds
      break
    }
  }
  if (is.na(dataset_name)) dataset_name <- "Unknown_Data"
  
  # 3. データ処理
  thresh_val <- thresholds[[target_var]]
  df <- read.csv(f_path, check.names = FALSE)
  
  # 必要な列があるか確認 (Type列必須)
  req_cols <- c("Observed", "Predicted", "SampleID", "Type")
  if (!all(req_cols %in% colnames(df))) next
  
  df$Residual <- df$Observed - df$Predicted
  df$AbsError <- abs(df$Residual)
  df$APE <- ifelse(df$Observed == 0, NA, (df$AbsError / abs(df$Observed)) * 100)
  
  df$Direction <- ifelse(df$Residual > 0, 
                         "Underestimated (Pred < Obs)", 
                         "Overestimated (Pred > Obs)")
  
  # 閾値超えのみ抽出
  bad_samples <- df %>% filter(AbsError > thresh_val)
  
  if (nrow(bad_samples) > 0) {
    temp_res <- data.frame(
      Model = model_name,
      Dataset = dataset_name,
      Target = target_var,
      SampleID = bad_samples$SampleID,
      Type = bad_samples$Type, # Typeを保持
      Observed = bad_samples$Observed,
      Predicted = bad_samples$Predicted,
      Residual = bad_samples$Residual,
      AbsError = bad_samples$AbsError,
      APE_Percent = round(bad_samples$APE, 2),
      Direction = bad_samples$Direction,
      SourceFile = fname
    )
    bad_list_all <- rbind(bad_list_all, temp_res)
  }
}

# ==============================================================================
# PHASE 2: 内装(Inner)と外挿(Outer)の分割と保存
# ==============================================================================

if (nrow(bad_list_all) > 0) {
  cat("\n=== Phase 2: Splitting Inner vs Outer ===\n")
  
  # --- 分割ロジック ---
  # Type == "OOD_Test" -> Outer (外挿)
  # それ以外 -> Inner (内装)
  
  bad_list_outer <- bad_list_all %>% filter(Type == "OOD_Test")
  bad_list_inner <- bad_list_all %>% filter(Type != "OOD_Test")
  
  cat(sprintf("Total Bad Samples: %d\n", nrow(bad_list_all)))
  cat(sprintf(" - Outer (OOD_Test): %d\n", nrow(bad_list_outer)))
  cat(sprintf(" - Inner (Others):   %d\n", nrow(bad_list_inner)))
  
  # 保存用関数の定義
  save_results <- function(data, prefix_label) {
    if (nrow(data) == 0) return(NULL)
    
    # 並べ替え
    data <- data %>% 
      select(SampleID, Type, Target, AbsError, APE_Percent, Direction, Model, Dataset, Observed, Predicted, Residual, SourceFile) %>%
      arrange(desc(AbsError))
    
    # ファイルパス生成
    file_detail <- file.path(output_dir, paste0(target_folder_date, "_", prefix_label, "_Integrated_Bad_Predictions.csv"))
    file_summary <- file.path(output_dir, paste0(target_folder_date, "_", prefix_label, "_Summary_Worst_SampleIDs.csv"))
    
    # 詳細保存
    write.csv(data, file_detail, row.names = FALSE)
    
    # 簡易集計保存
    summary_table <- data %>%
      group_by(SampleID) %>%
      summarise(
        Total_Failures = n(),
        Mean_APE = mean(APE_Percent, na.rm=TRUE),
        Fail_Types = paste(unique(Type), collapse=", "), # 確認用
        Models = paste(unique(Model), collapse=", ")
      ) %>%
      arrange(desc(Total_Failures))
    
    write.csv(summary_table, file_summary, row.names = FALSE)
    return(file_detail) # 解析用にパスを返す
  }
  
  # 保存実行
  path_inner <- save_results(bad_list_inner, "Inner")
  path_outer <- save_results(bad_list_outer, "Outer")
  
} else {
  stop("No bad predictions found in any files.")
}

# ==============================================================================
# PHASE 3: 詳細トレンド分析 (Inner / Outer 個別に実行)
# ==============================================================================

cat("\n=== Phase 3: Detailed Trend Analysis ===\n")

# 分析ロジックを関数化
analyze_trends_func <- function(input_csv_path, label) {
  
  if (is.null(input_csv_path)) {
    cat(sprintf("[%s] No data to analyze.\n", label))
    return()
  }
  
  cat(sprintf("Analyzing [%s] data...\n", label))
  df_worst <- read.csv(input_csv_path, check.names = FALSE)
  
  # --- ランク付け ---
  worst_ranking <- df_worst %>% count(SampleID) %>% arrange(desc(n))
  
  # --- 視点1: Target ---
  cross_target <- df_worst %>%
    count(SampleID, Target) %>%
    pivot_wider(names_from = Target, values_from = n, values_fill = 0, names_prefix = "Tgt_")
  
  target_cols <- grep("^Tgt_", names(cross_target), value = TRUE)
  cross_target$Trend_Target <- apply(cross_target[, target_cols, drop=FALSE], 1, function(x) {
    raw_names <- gsub("^Tgt_", "", names(x))
    cols <- raw_names[x > 0]
    if (length(cols) == 4) return("All Targets")
    if ("PCEmax" %in% cols && length(cols) == 1) return("PCE Only")
    return(paste(cols, collapse = "+"))
  })
  
  # --- 視点2: Model ---
  cross_model <- df_worst %>%
    count(SampleID, Model) %>%
    pivot_wider(names_from = Model, values_from = n, values_fill = 0, names_prefix = "Mod_")
  
  linear_models <- c("PLS", "svmLinear")
  nonlinear_models <- c("MonMLP", "SVMRadial", "gaussPrRadial", "Rborist", "gcvEarth", "ppr", "monmlp", "knn")
  model_cols <- grep("^Mod_", names(cross_model), value = TRUE)
  
  cross_model$Trend_Model <- apply(cross_model[, model_cols, drop=FALSE], 1, function(x) {
    raw_names <- gsub("^Mod_", "", names(x))
    failed <- raw_names[x > 0]
    n_lin <- sum(failed %in% linear_models)
    n_non <- sum(failed %in% nonlinear_models)
    if (length(failed) >= 8) return("All Models")
    if (n_lin > 0 && n_non == 0) return("Linear Only")
    return("Mixed")
  })
  
  # --- 視点3: Dataset ---
  cross_dataset <- df_worst %>%
    count(SampleID, Dataset) %>%
    pivot_wider(names_from = Dataset, values_from = n, values_fill = 0, names_prefix = "Dat_")
  
  dataset_cols <- grep("^Dat_", names(cross_dataset), value = TRUE)
  cross_dataset$Trend_Dataset <- apply(cross_dataset[, dataset_cols, drop=FALSE], 1, function(x) {
    raw_names <- gsub("^Dat_", "", names(x))
    failed_ds <- raw_names[x > 0]
    n_failed <- length(failed_ds)
    if (n_failed >= 10) return("All Datasets")
    has_base <- any(str_detect(failed_ds, "base"))
    has_all  <- any(str_detect(failed_ds, "all"))
    if (has_base && !has_all) return("Base Only (All fixed)")
    has_oh <- any(str_detect(failed_ds, "OH"))
    has_fp <- any(str_detect(failed_ds, "FP"))
    if (has_fp && !has_oh) return("FP Only (OH fixed)")
    if (has_oh && !has_fp) return("OH Only (FP fixed)")
    return("Mixed / Specific")
  })
  
  # --- 視点4: Direction ---
  df_worst$Dir_Simp <- ifelse(str_detect(df_worst$Direction, "Over"), "Over", "Under")
  cross_dir <- df_worst %>%
    count(SampleID, Dir_Simp) %>%
    pivot_wider(names_from = Dir_Simp, values_from = n, values_fill = 0)
  
  cross_dir$Trend_Dir <- apply(cross_dir[, -1, drop=FALSE], 1, function(x) {
    over <- ifelse(is.na(x["Over"]), 0, x["Over"])
    under <- ifelse(is.na(x["Under"]), 0, x["Under"])
    if (over == (over + under)) return("Mostly Over")
    if (under == (over + under)) return("Mostly Under")
    if (over > under * 2) return("Mostly Over")
    if (under > over * 2) return("Mostly Under")
    return("Unstable")
  })
  
  # --- 統合 ---
  final_report <- worst_ranking %>%
    left_join(cross_target, by = "SampleID") %>%
    left_join(cross_model, by = "SampleID") %>%
    left_join(cross_dataset, by = "SampleID") %>%
    left_join(cross_dir %>% select(SampleID, Trend_Dir), by = "SampleID") %>%
    select(
      SampleID, n, 
      Trend_Target, Trend_Model, Trend_Dataset, Trend_Dir,
      starts_with("Tgt_"),
      starts_with("Mod_"),
      starts_with("Dat_")
    )
  
  # 保存
  output_file <- file.path(output_dir, paste0(target_folder_date, "_", label, "_Analysis_Trend_Report.csv"))
  write.csv(final_report, output_file, row.names = FALSE)
  cat(sprintf(" -> Saved report: %s\n", basename(output_file)))
}

# 実行
analyze_trends_func(path_inner, "Inner")
analyze_trends_func(path_outer, "Outer")

cat("\n=== All Processes Completed ===\n")
cat(sprintf("Results are located in: %s\n", output_dir))

=== Phase 1: Scanning and Aggregating Data ===
Found 480 files. Processing...

=== Phase 2: Splitting Inner vs Outer ===
Total Bad Samples: 37452
 - Outer (OOD_Test): 12137
 - Inner (Others):   25315

=== Phase 3: Detailed Trend Analysis ===
Analyzing [Inner] data...
 -> Saved report: 20251217_Inner_Analysis_Trend_Report.csv
Analyzing [Outer] data...
 -> Saved report: 20251217_Outer_Analysis_Trend_Report.csv

=== All Processes Completed ===
Results are located in: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression//20251217_Analysis_Split_Inner_Outer


In [19]:
# ==============================================================================
# 解析視点の転換：モデル別・ターゲット別のパフォーマンス比較 (Inner vs Outer)
# ==============================================================================

library(dplyr)
library(tidyr)
library(stringr)

# --- 設定 ---
root_dir <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/"
target_folder_date <- "20251217"
output_dir <- file.path(root_dir, paste0(target_folder_date, "_Analysis_Split_Inner_Outer"))

# --- 読み込み関数 ---
load_data <- function(label) {
  # ファイル名パターン
  pat <- paste0(target_folder_date, "_", label, "_Integrated_Bad_Predictions.csv")
  files <- list.files(path = output_dir, pattern = pat, full.names = TRUE)
  
  if (length(files) == 0) return(NULL)
  
  df <- read.csv(files[1], check.names = FALSE)
  df$Region <- label # Inner か Outer かのタグ付け
  return(df)
}

# データのロード
df_inner <- load_data("Inner")
df_outer <- load_data("Outer")

if (is.null(df_inner) && is.null(df_outer)) stop("データが見つかりません。先ほどの分割処理を完了してください。")

# データを結合（比較しやすくするため）
df_all <- bind_rows(df_inner, df_outer)

# ==============================================================================
# 視点1: Target別の難易度分析
# ==============================================================================
target_summary <- df_all %>%
  group_by(Region, Target) %>%
  summarise(
    Fail_Count = n(),
    Mean_APE = mean(APE_Percent, na.rm = TRUE),
    Mean_AbsError = mean(AbsError, na.rm = TRUE),
    # 方向性の割合 (Overの割合)
    Over_Ratio = mean(str_detect(Direction, "Over")) * 100
  ) %>%
  mutate(
    Over_Under_Trend = ifelse(Over_Ratio > 60, "Mostly Over",
                       ifelse(Over_Ratio < 40, "Mostly Under", "Balanced"))
  ) %>%
  arrange(Region, desc(Fail_Count))

# --- 保存 ---
write.csv(target_summary, file.path(output_dir, "Comparison_Target_Performance.csv"), row.names = FALSE)

# ==============================================================================
# 視点2: Model別の堅牢性分析
# ==============================================================================
model_summary <- df_all %>%
  group_by(Region, Model) %>%
  summarise(
    Fail_Count = n(),
    Mean_APE = mean(APE_Percent, na.rm = TRUE),
    # 失敗の深刻度（エラーの平均値）
    Mean_AbsError = mean(AbsError, na.rm = TRUE),
    # バイアス (Overの割合)
    Over_Ratio = mean(str_detect(Direction, "Over")) * 100
  ) %>%
  mutate(
    Over_Under_Trend = ifelse(Over_Ratio > 60, "Over Bias",
                       ifelse(Over_Ratio < 40, "Under Bias", "Neutral"))
  ) %>%
  arrange(Region, desc(Fail_Count))

# --- 保存 ---
write.csv(model_summary, file.path(output_dir, "Comparison_Model_Performance.csv"), row.names = FALSE)

# ==============================================================================
# コンソールへ簡易レポート出力
# ==============================================================================
cat("\n=== Target Analysis (Top 3 Hardest) ===\n")
print(head(target_summary, 6))

cat("\n=== Model Analysis (Top 5 Most Failures) ===\n")
print(head(model_summary, 10))

cat(sprintf("\nAnalysis saved to: %s\n", output_dir))

`summarise()` has grouped output by 'Region'. You can override using the `.groups` argument.
`summarise()` has grouped output by 'Region'. You can override using the `.groups` argument.



=== Target Analysis (Top 3 Hardest) ===
# A tibble: 6 x 7
# Groups:   Region [2]
  Region Target Fail_Count Mean_APE Mean_AbsError Over_Ratio Over_Under_Trend
  <chr>  <chr>       <int>    <dbl>         <dbl>      <dbl> <chr>           
1 Inner  Jsc          6866    130.          3.23        50.0 Balanced        
2 Inner  Voc          6822     55.7         0.161       53.1 Balanced        
3 Inner  PCEmax       6320    551.          1.65        44.3 Balanced        
4 Inner  FF           5307     32.8         0.133       54.3 Balanced        
5 Outer  Jsc          3625    219.          6.65        52.5 Balanced        
6 Outer  Voc          3215     91.8         0.358       50.7 Balanced        

=== Model Analysis (Top 5 Most Failures) ===
# A tibble: 10 x 7
# Groups:   Region [2]
   Region Model    Fail_Count Mean_APE Mean_AbsError Over_Ratio Over_Under_Trend
   <chr>  <chr>         <int>    <dbl>         <dbl>      <dbl> <chr>           
 1 Inner  monmlp         7715    160.       

In [22]:
# ==============================================================================
# PLS専用 16条件集計コード (検索ロジック修正版)
# ==============================================================================

library(dplyr)
library(stringr)
library(caret)

# --- 1. 設定 ---
root_dir <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/"
target_date <- "20251217"
target_model <- "PLS" 

# 軸の定義
target_datasets <- c("base", "base_OH", "base_FP", "all", "all_OH", "all_FP")
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
imputations <- c("n", "m") 
regions <- c("Inner", "Outer")

# --- 2. フォルダとファイル取得 ---

model_base_dir <- file.path(root_dir, target_model)
if (!dir.exists(model_base_dir)) stop(paste("Model dir not found:", model_base_dir))

sub_dirs <- list.dirs(model_base_dir, full.names = TRUE, recursive = FALSE)
target_dir <- sub_dirs[str_detect(basename(sub_dirs), target_date)]

if (length(target_dir) == 0) stop("Target date folder not found.")
target_dir <- target_dir[1] # 複数あれば最初を使用

cat(sprintf("Target Folder: %s\n", basename(target_dir)))
all_files <- list.files(path = target_dir, pattern = "_pred.csv$", full.names = TRUE)
cat(sprintf("Found %d CSV files. \n", length(all_files)))

# ★デバッグ：実際のファイル名を5つ表示して確認
cat("--- Example Filenames ---\n")
print(head(basename(all_files), 5))
cat("-----------------------\n")

# --- 3. フォーマット関数 ---

format_metrics <- function(obs, pred) {
  if (length(obs) < 2) return("N/A")
  r2_val <- caret::R2(pred, obs, na.rm = TRUE)
  rmse_val <- caret::RMSE(pred, obs, na.rm = TRUE)
  return(sprintf("%.3f (%.3f)", r2_val, rmse_val))
}

format_sci <- function(val) {
  if (is.na(val) || val == "") return("")
  # 文字列から数値抽出を試みる
  num_val <- as.numeric(str_extract(val, "[0-9\\.]+"))
  if (is.na(num_val)) return(val)
  
  if (num_val < 0.01 && num_val > 0) {
    return(formatC(num_val, format = "e", digits = 2))
  } else {
    return(as.character(num_val))
  }
}

# --- 4. 集計ループ ---

cat("\n=== PLS Summary Table (16 Conditions) ===\n")

for (imp in imputations) {       
  for (tgt in target_vars) {     
    for (reg in regions) {       
      
      condition_label <- paste0("[", imp, "_", tgt, "] ", reg)
      
      row_metrics <- c("Condition" = condition_label, "Item" = "Score")
      row_params  <- c("Condition" = "", "Item" = "Param")
      
      for (ds_key in target_datasets) {
        
        # --- 厳密なフィルタリング ---
        # 1. 補完: 先頭(^) または _ の直後にある n_ / m_ を探す
        # 2. ターゲット: _Jsc_ など
        # 3. データセット: baseは含むが OH/FP は含まない、等の除外処理
        
        subset_files <- all_files[
          str_detect(basename(all_files), paste0("(^|_)", imp, "_")) &
          str_detect(basename(all_files), paste0("_", tgt, "_"))
        ]
        
        # データセット判定
        if (ds_key == "base") {
          subset_files <- subset_files[str_detect(basename(subset_files), "_base_") & !str_detect(basename(subset_files), "_OH_|_FP_")]
        } else if (ds_key == "all") {
          subset_files <- subset_files[str_detect(basename(subset_files), "_all_") & !str_detect(basename(subset_files), "_OH_|_FP_")]
        } else {
          # base_OH, all_FP などはそのまま検索
          subset_files <- subset_files[str_detect(basename(subset_files), paste0("_", ds_key, "_"))]
        }
        
        val_cell <- "Missing"
        param_cell <- "" 
        
        if (length(subset_files) == 1) {
          f_path <- subset_files[1]
          df <- read.csv(f_path, check.names = FALSE)
          
          if ("Type" %in% colnames(df)) {
            if (reg == "Outer") {
              df_sub <- df %>% filter(Type == "OOD_Test")
            } else {
              df_sub <- df %>% filter(Type != "OOD_Test")
            }
            
            if (nrow(df_sub) > 0) {
              val_cell <- format_metrics(df_sub$Observed, df_sub$Predicted)
            } else {
              val_cell <- "No Samples"
            }
            
            # ハイパラ取得 (ファイル名から comp-XX を抽出)
            # 例: 2025..._PLS_comp-10_... -> "10"
            p_match <- str_extract(basename(f_path), "comp-[0-9]+")
            if (!is.na(p_match)) {
              param_cell <- str_replace(p_match, "comp-", "n_comp=")
            }
            
          } else {
            val_cell <- "Error(NoType)"
          }
        } else if (length(subset_files) > 1) {
          val_cell <- "Dupe" # ここが出たらフィルタ条件不足
        }
        
        row_metrics[ds_key] <- val_cell
        row_params[ds_key]  <- param_cell
      }
      
      print(bind_rows(row_metrics, row_params))
    }
  }
}

Target Folder: 20251217_PLS_12Files_Results
Found 48 CSV files. 
--- Example Filenames ---
[1] "fixed_20251217_PLS_m_all_FF_pred.csv"               
[2] "fixed_20251217_PLS_m_all_FP_rebuilt_FF_pred.csv"    
[3] "fixed_20251217_PLS_m_all_FP_rebuilt_Jsc_pred.csv"   
[4] "fixed_20251217_PLS_m_all_FP_rebuilt_PCEmax_pred.csv"
[5] "fixed_20251217_PLS_m_all_FP_rebuilt_Voc_pred.csv"   
-----------------------

=== PLS Summary Table (16 Conditions) ===
# A tibble: 2 x 8
  Condition       Item  base            base_OH      base_FP all   all_OH all_FP
  <chr>           <chr> <chr>           <chr>        <chr>   <chr> <chr>  <chr> 
1 "[n_Jsc] Inner" Score "0.938 (1.181)" "0.984 (0.6~ "0.938~ "0.9~ "0.98~ "0.94~
2 ""              Param ""              ""           ""      ""    ""     ""    
# A tibble: 2 x 8
  Condition       Item  base            base_OH      base_FP all   all_OH all_FP
  <chr>           <chr> <chr>           <chr>        <chr>   <chr> <chr>  <chr> 
1 "[n_Jsc] Outer" Score "0.957

In [27]:
# ==============================================================================
# Final Tool: Parameter Expanded with Scientific Notation
# ==============================================================================

library(dplyr)
library(stringr)
library(tidyr)

# --- 1. 設定 ---
root_dir <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/"
target_date <- "20251217"
output_dir <- file.path(root_dir, paste0(target_date, "_Final_Summary_Tables_Formatted"))

if (!dir.exists(output_dir)) dir.create(output_dir)

model_order <- c("PLS", "svmLinear", "gaussprLinear", "svmRadial", "gaussprRadial", "gcvEarth", "ppr", "knn", "Rborist", "monmlp")

model_folder_keywords <- list(
  "PLS"           = "PLS",
  "svmLinear"     = c("svmLinear", "SVMLinear"),
  "gaussprLinear" = c("gaussprLinear", "GPRLinear"),
  "svmRadial"     = c("svmRadial", "SVMRadial", "SVM"),
  "gaussprRadial" = c("gaussprRadial", "GPR"),
  "gcvEarth"      = "gcvEarth",
  "ppr"           = "ppr",
  "knn"           = "knn",
  "Rborist"       = "Rborist",
  "monmlp"        = "monmlp"
)

target_datasets <- c("base", "base_OH", "base_FP", "all", "all_OH", "all_FP")
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
imputations <- c("n", "m")
regions <- c("Inner", "Outer")

# --- 2. 補助関数 ---

read_model_summary <- function(mod_name) {
  keywords <- model_folder_keywords[[mod_name]]
  if (is.null(keywords)) keywords <- mod_name
  sub_dirs <- list.dirs(root_dir, full.names = TRUE, recursive = FALSE)
  
  target_model_dir <- NULL
  for (kw in keywords) {
    hits <- sub_dirs[str_detect(basename(sub_dirs), fixed(kw, ignore_case = TRUE))]
    if (length(hits) > 0) { target_model_dir <- hits[1]; break }
  }
  if (is.null(target_model_dir)) return(NULL)
  
  deep_dirs <- list.dirs(target_model_dir, full.names = TRUE, recursive = FALSE)
  date_dir <- deep_dirs[str_detect(basename(deep_dirs), target_date)]
  if (length(date_dir) == 0) return(NULL)
  
  sum_file <- list.files(date_dir[1], pattern = "summary.csv", full.names = TRUE)
  if (length(sum_file) == 0) return(NULL)
  
  return(read.csv(sum_file[1], stringsAsFactors = FALSE))
}

# --- ★新規追加: 科学的表記への変換関数 ---
format_scientific_custom <- function(val_str) {
  # 数値として解釈できるか試す
  num_val <- suppressWarnings(as.numeric(val_str))
  
  # 数値ではない、または0、または0.001以上の場合はそのまま返す
  if (is.na(num_val) || num_val == 0 || abs(num_val) >= 0.001) {
    return(val_str)
  }
  
  # 科学的表記に変換 (例: 1.456e-05)
  # digits=3 で小数点以下3桁 (1.456) まで確保
  sci_str <- formatC(num_val, format = "e", digits = 3)
  
  # "e" を分解して "x 10^" 形式に整形
  parts <- str_split(sci_str, "e")[[1]]
  base <- parts[1]
  exponent <- as.numeric(parts[2]) # 不要な0を削除 (-05 -> -5)
  
  # 形式: 1.456 x 10^-5
  # CSVで文字化けしないよう、標準的な ASCII 文字で構成します
  return(sprintf("%s x 10^%d", base, exponent))
}

parse_params_to_df <- function(param_str) {
  if (is.na(param_str) || param_str == "" || param_str == "None") {
    return(data.frame(Key = character(0), Value = character(0)))
  }
  
  clean_str <- str_replace_all(param_str, "[\\{\\}\\'\"]", "")
  
  # monmlpのタプル対策
  while(str_detect(clean_str, "\\([^\\)]*,[^\\)]*\\)")) {
    clean_str <- str_replace(clean_str, "(\\([^\\)]*),([^\\)]*\\))", "\\1;\\2")
  }
  
  pairs <- str_split(clean_str, ",")[[1]]
  keys <- c()
  values <- c()
  
  for (p in pairs) {
    if (str_detect(p, ":")) {
      parts <- str_split(p, ":", n=2)[[1]]
    } else if (str_detect(p, "=")) {
      parts <- str_split(p, "=", n=2)[[1]]
    } else {
      next
    }
    
    k <- str_trim(parts[1])
    raw_v <- str_trim(parts[2])
    raw_v <- str_replace_all(raw_v, ";", ",")
    
    # ★ここで値を整形
    formatted_v <- format_scientific_custom(raw_v)
    
    keys <- c(keys, k)
    values <- c(values, formatted_v)
  }
  
  return(data.frame(Key = keys, Value = values, stringsAsFactors = FALSE))
}

# --- 3. メイン処理 ---

cat("=== Starting Batch Processing (Formatted) ===\n")
all_summaries <- list()
for (mod in model_order) { all_summaries[[mod]] <- read_model_summary(mod) }

count <- 1
total_tasks <- length(imputations) * length(target_vars) * length(regions)

for (imp in imputations) {
  for (tgt in target_vars) {
    for (reg in regions) {
      
      file_name <- sprintf("Summary_%s_%s_%s.csv", reg, imp, tgt)
      cat(sprintf("[%d/%d] Generating: %s ... ", count, total_tasks, file_name))
      
      final_rows <- list()
      
      for (mod in model_order) {
        df <- all_summaries[[mod]]
        
        score_row <- c(Model = mod)
        params_storage <- list()
        all_potential_keys <- c()
        
        for (ds in target_datasets) {
          val_cell <- "Missing"
          parsed_params <- data.frame(Key = character(0), Value = character(0))
          
          if (!is.null(df)) {
            target_rows <- df %>% filter(Target == tgt, str_detect(File, paste0("(^|_)", imp, "_")))
            
            if (ds == "base") {
              target_rows <- target_rows %>% filter(str_detect(File, "_base") & !str_detect(File, "OH") & !str_detect(File, "FP"))
            } else if (ds == "all") {
              target_rows <- target_rows %>% filter(str_detect(File, "_all") & !str_detect(File, "OH") & !str_detect(File, "FP"))
            } else {
              target_rows <- target_rows %>% filter(str_detect(File, paste0("_", ds)))
            }
            
            if (nrow(target_rows) > 0) {
              r <- target_rows[1, ]
              
              r2 <- if(reg=="Inner") r$R2 else r$OOD_R2
              rmse <- if(reg=="Inner") r$RMSE else r$OOD_RMSE
              
              if (!is.na(r2) && !is.na(rmse)) {
                val_cell <- sprintf("%.3f (%.3f)", as.numeric(r2), as.numeric(rmse))
              } else {
                val_cell <- "N/A"
              }
              parsed_params <- parse_params_to_df(r$Best_Params)
            }
          }
          
          score_row[ds] <- val_cell
          params_storage[[ds]] <- parsed_params
          all_potential_keys <- unique(c(all_potential_keys, parsed_params$Key))
        }
        
        final_rows[[length(final_rows) + 1]] <- score_row
        
        if (length(all_potential_keys) > 0) {
          for (pkey in all_potential_keys) {
            param_row <- c(Model = paste0("(", pkey, ")"))
            for (ds in target_datasets) {
              p_df <- params_storage[[ds]]
              if (nrow(p_df) > 0 && pkey %in% p_df$Key) {
                param_row[ds] <- p_df$Value[p_df$Key == pkey]
              } else {
                param_row[ds] <- ""
              }
            }
            final_rows[[length(final_rows) + 1]] <- param_row
          }
        }
      }
      
      final_df <- bind_rows(lapply(final_rows, function(x) as.data.frame(t(x), stringsAsFactors=FALSE)))
      
      cols_order <- c("Model", target_datasets)
      final_df <- final_df[, cols_order]
      
      write.csv(final_df, file.path(output_dir, file_name), row.names = FALSE)
      cat("Done.\n")
      count <- count + 1
    }
  }
}

cat("\n=== All Formatted CSVs Created ===\n")
cat(output_dir, "\n")

=== Starting Batch Processing (Formatted) ===
[1/16] Generating: Summary_Inner_n_Jsc.csv ... Done.
[2/16] Generating: Summary_Outer_n_Jsc.csv ... Done.
[3/16] Generating: Summary_Inner_n_Voc.csv ... Done.
[4/16] Generating: Summary_Outer_n_Voc.csv ... Done.
[5/16] Generating: Summary_Inner_n_FF.csv ... Done.
[6/16] Generating: Summary_Outer_n_FF.csv ... Done.
[7/16] Generating: Summary_Inner_n_PCEmax.csv ... Done.
[8/16] Generating: Summary_Outer_n_PCEmax.csv ... Done.
[9/16] Generating: Summary_Inner_m_Jsc.csv ... Done.
[10/16] Generating: Summary_Outer_m_Jsc.csv ... Done.
[11/16] Generating: Summary_Inner_m_Voc.csv ... Done.
[12/16] Generating: Summary_Outer_m_Voc.csv ... Done.
[13/16] Generating: Summary_Inner_m_FF.csv ... Done.
[14/16] Generating: Summary_Outer_m_FF.csv ... Done.
[15/16] Generating: Summary_Inner_m_PCEmax.csv ... Done.
[16/16] Generating: Summary_Outer_m_PCEmax.csv ... Done.

=== All Formatted CSVs Created ===
/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_P